# ChromaDB + Keyword Extraction Demo

In [ ]:
# !pip install chromadb
# !pip install keybert

In [ ]:
from pprint import pprint

## Corpus

In [ ]:
# Neurogenesis - Wikipedia
document_1 = """
Neurogenesis is the process by which nervous system cells, the neurons, are produced by neural stem cells (NSCs).
This occurs in all species of animals except the porifera (sponges) and placozoans.
Types of NSCs include neuroepithelial cells (NECs), radial glial cells (RGCs), basal progenitors (BPs), 
intermediate neuronal precursors (INPs), subventricular zone astrocytes, and subgranular zone radial astrocytes, among others.
Neurogenesis is most active during embryonic development and is responsible for producing all the various types of neurons 
of the organism, but it continues throughout adult life in a variety of organisms. Once born, neurons do not divide (see mitosis), 
and many will live the lifespan of the animal, except under extraordinary and usually pathogenic circumstances.
"""

# Jet engines - Wikipedia
document_2 = """
A jet engine is a type of reaction engine, discharging a fast-moving jet of heated gas (usually air) that generates thrust by jet propulsion. 
While this broad definition may include rocket, water jet, and hybrid propulsion, the term jet engine typically refers to an internal combustion 
air-breathing jet engine such as a turbojet, turbofan, ramjet, pulse jet, or scramjet. In general, jet engines are internal combustion engines.
Air-breathing jet engines typically feature a rotating air compressor powered by a turbine, with the leftover power providing thrust through 
the propelling nozzle—this process is known as the Brayton thermodynamic cycle. Jet aircraft use such engines for long-distance travel. 
Early jet aircraft used turbojet engines that were relatively inefficient for subsonic flight. Most modern subsonic jet aircraft use more 
complex high-bypass turbofan engines. They give higher speed and greater fuel efficiency than piston and propeller aeroengines over long distances.
A few air-breathing engines made for high-speed applications (ramjets and scramjets) use the ram effect of the vehicle's speed instead of 
a mechanical compressor.
"""

# Jet engines - NASA
document_3 = """
All jet engines, which are also called gas turbines, work on the same principle. The engine sucks air in at the front with a fan. 
A compressor raises the pressure of the air. The compressor is made with many blades attached to a shaft. The blades spin at high speed and compress 
or squeeze the air. The compressed air is then sprayed with fuel and an electric spark lights the mixture. The burning gases expand and blast out 
through the nozzle, at the back of the engine. As the jets of gas shoot backward, the engine and the aircraft are thrust forward. 
As the hot air is going to the nozzle, it passes through another group of blades called the turbine. The turbine is attached to the same shaft as 
the compressor. Spinning the turbine causes the compressor to spin.
"""

# Tesla - company
document_4 = """
Tesla, Inc. is an American multinational automotive and clean energy company. Headquartered in Austin, Texas, it designs, manufactures and 
sells battery electric vehicles (BEVs), stationary battery energy storage devices from home to grid-scale, solar panels and solar shingles, 
and related products and services.
"""

# Tesla - person
document_5 = """
Nikola Tesla (10 July 1856 – 7 January 1943) was a Serbian-American engineer, futurist, and inventor. He is known for his contributions 
to the design of the modern alternating current (AC) electricity supply system.
"""

DOCUMENTS = [document_1, document_2, document_3, document_4, document_5]

## Keyword Extraction

Although not useful with `ChromaDB` (it doesn't support lists as metadata values, nor "contains" operator on strings), you might find this useful with other Vector Stores.

In [ ]:
from keybert import KeyBERT

In [ ]:
KW_MODEL = KeyBERT(model="all-MiniLM-L6-v2")

In [ ]:
doc_1_kws = KW_MODEL.extract_keywords(document_1, top_n=5, keyphrase_ngram_range=(1, 2))
doc_1_kws

In [ ]:
doc_2_kws = KW_MODEL.extract_keywords(document_2, top_n=5, keyphrase_ngram_range=(1, 2))
doc_2_kws

In [ ]:
doc_3_kws = KW_MODEL.extract_keywords(document_3, top_n=5, keyphrase_ngram_range=(1, 2))
doc_3_kws

In [ ]:
doc_4_kws = KW_MODEL.extract_keywords(document_4, top_n=5, keyphrase_ngram_range=(1, 2))
doc_4_kws

In [ ]:
doc_5_kws = KW_MODEL.extract_keywords(document_5, top_n=5, keyphrase_ngram_range=(1, 2))
doc_5_kws

Prepare a plain list of keywords per document, to tag the document with when put in the vector store:

In [ ]:
doc_kws = [doc_1_kws, doc_2_kws, doc_3_kws, doc_4_kws, doc_5_kws]

In [ ]:
def process_kws(doc_kws: list[tuple[str, float]], threshold: float = 0.5) -> list[str]:
    return [
        kw
        for kw, score in doc_kws
    ]

In [ ]:
processed_doc_kws = [
    process_kws(single_doc_kws)
    for single_doc_kws in doc_kws 
]

processed_doc_kws

## Vector Database

ChromaDB gives us the option to either provide our own embeddings, or let it take care of it for us when a document is added to the database. Here, we'll use the latter and let ChromaDB do the work for us.

In [ ]:
import hashlib

import chromadb

In [ ]:
CHROMADB_CLIENT = chromadb.Client()

In [ ]:
CHROMADB_COLLECTION_L2 = CHROMADB_CLIENT.get_or_create_collection(
    name="augmentation-docs-1",
    embedding_function=chromadb.utils.embedding_functions.ONNXMiniLM_L6_V2()
)

In [ ]:
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

mpnet_ef = SentenceTransformerEmbeddingFunction(model_name="all-mpnet-base-v2")

CHROMADB_COLLECTION_COSINE = CHROMADB_CLIENT.get_or_create_collection(
    name="augmentation-docs-2",
    embedding_function=mpnet_ef,
    configuration={    # L2 is the default
        "hnsw": {
            "space": "cosine"
        },
    }
)

In [ ]:
for doc_index, document in enumerate(DOCUMENTS):
    
    CHROMADB_COLLECTION_L2.add(
            ids=[hashlib.md5(document.encode()).hexdigest()[-8:]],  # Unique identifier
            documents=[document],  # Text for embedding
            metadatas=[{
                "word-count": len(document.split()),
                "source": "wikipedia",
                "kws": processed_doc_kws[doc_index]
            }],
    )

    CHROMADB_COLLECTION_COSINE.add(
            ids=[hashlib.md5(document.encode()).hexdigest()[-8:]],  # Unique identifier
            documents=[document],  # Text for embedding
            metadatas=[{
                "word-count": len(document.split()),
                "source": "wikipedia",
                "kws": processed_doc_kws[doc_index]
            }],
    )

In [ ]:
# for document in DOCUMENTS:
    
#     CHROMADB_COLLECTION_L2.add(
#             ids=[hashlib.md5(document.encode()).hexdigest()[-8:]],  # Unique identifier
#             documents=[document],  # Text for embedding
#             metadatas=[{
#                 "word-count": len(document.split()),
#                 "source": "wikipedia"
#             }],
#     )

#     CHROMADB_COLLECTION_COSINE.add(
#             ids=[hashlib.md5(document.encode()).hexdigest()[-8:]],  # Unique identifier
#             documents=[document],  # Text for embedding
#             metadatas=[{
#                 "word-count": len(document.split()),
#                 "source": "wikipedia"
#             }],
#     )

In [ ]:
CHROMADB_COLLECTION_L2.count()

#### Full-text search

In [ ]:
text_search_results = CHROMADB_COLLECTION_L2.get(
    include=["documents", "metadatas"],
    where={                           # metadata
        "source": "wikipedia",
    },
    where_document={                  # document
        "$or": [
            {"$contains": "jet engine"},
            {"$contains": "turbine"},
        ]
    }
)

In [ ]:
len(text_search_results["ids"])

In [ ]:
pprint(text_search_results)

#### Filtering

In [ ]:
filtered_results = CHROMADB_COLLECTION_L2.get(
    include=["documents", "metadatas"],
    where={                           # metadata
        "$and": [
            {"source": "wikipedia"},
            {"kws": {"$contains": "energy company"}}
        ]
    }
)

In [ ]:
len(filtered_results["documents"])

In [ ]:
filtered_results["documents"][0]

#### Similarity Search

L2 (Euclidean):

In [ ]:
search_results_l2 = CHROMADB_COLLECTION_L2.query(
    query_texts=["Is it possible for brain cells to regenerate?"],  # Text for similarity search
    n_results=2,
    where={"source": "wikipedia"},  
    include=["documents", "distances"],
)

In [ ]:
# Lower is better
pprint(list(zip(search_results_l2["distances"][0], search_results_l2["documents"][0])))

Cosine Similarity:

In [ ]:
search_results_cosine = CHROMADB_COLLECTION_COSINE.query(
    query_texts=["Is it possible for brain cells to regenerate?"],  # Text for similarity search
    n_results=2,
    where={"source": "wikipedia"},  
    include=["documents", "distances"],
)

ChromaDB returns `(1 - cosine)` instead of just `cosine` (why?). See: https://docs.trychroma.com/docs/collections/configure

In [ ]:
pprint(list(zip(search_results_cosine["distances"][0], search_results_cosine["documents"][0])))

<hr/>